In [1]:
import pandas as pd
import re

In [2]:
# logic.py
"""
Python port of your loan suggestion logic from logic.js.
Functions:
  - update_with_la(records, la)
  - update_with_da(records, da)
  - query_complex(scenarios, deposit_amount=None, repayment_duration=None,
                  deposit_duration=None, interest_rate=None, credit_score=None)
  - calculate_sort_order(loan)

Expect each record to be a dict with fields matching your JS schema.
Requires a JSON file 'parameters_weights.json' in the same directory.
"""
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def _load_weights() -> Dict[str, Any]:
    with open('parameters_weights.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_parameters_weights = _load_weights()


In [3]:
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def _load_record() -> Dict[str, Any]:
    with open('MEC-LoanRecomn_Scenarios-V15.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_records = _load_record()


In [4]:
_records

[{'id': 1,
  'nickname': 'بهان(بدون ضامن)',
  'package_name': 'فرابانک',
  'contract_type': 'مرابحه',
  'granted_method': 'واریز به حساب',
  'loan_amount_limit': 1000000000,
  'deposit_duration': 3,
  'interest_rate': 23,
  'repayment_duration': 60,
  'loan_coefficient': 1000,
  'credit_score': 'A',
  'minimum_deposit_amount': 'nan',
  'maximum_deposit_amount': 'nan',
  'minimum_loan_amount': 100000000,
  'guarantee': 'خیر',
  'receiving_channel': 'غیر حضوری'},
 {'id': 2,
  'nickname': 'بهان(بدون ضامن)',
  'package_name': 'فرابانک',
  'contract_type': 'مرابحه',
  'granted_method': 'واریز به حساب',
  'loan_amount_limit': 1000000000,
  'deposit_duration': 3,
  'interest_rate': 23,
  'repayment_duration': 60,
  'loan_coefficient': 1000,
  'credit_score': 'B',
  'minimum_deposit_amount': 'nan',
  'maximum_deposit_amount': 'nan',
  'minimum_loan_amount': 100000000,
  'guarantee': 'خیر',
  'receiving_channel': 'غیر حضوری'},
 {'id': 3,
  'nickname': 'شایان',
  'package_name': 'شایان یک',
  'c

In [5]:
def calculate_sort_order(loan: Dict[str, Any]) -> None:
    """
    Mutates loan by adding a 'sortOrder' key based on weighted criteria,
    faithfully mirroring the JS logic.
    """
    # Determine bucket
    loanAmountKey = 'out_of_range'
    la = loan.get('loan_amount', 0)
    if la <= 500_000_000:
        loanAmountKey = '1-50'
    elif la <= 1_000_000_000:
        loanAmountKey = '50-100'
    elif la <= 1_500_000_000:
        loanAmountKey = '100-150'
    elif la <= 2_000_000_000:
        loanAmountKey = '150-200'
    elif la <= 2_500_000_000:
        loanAmountKey = '200-250'
    elif la <= 3_000_000_000:
        loanAmountKey = '250-300'

    pw = _parameters_weights
    # Extract individual weights
    ir_key = str(loan.get('interest_rate', ''))
    rd_key = str(loan.get('repayment_duration', ''))
    dd_key = str(loan.get('deposit_duration', ''))
    # CS: first A-E or N
    cs_match = re.search(r'[ABCDE]', loan.get('credit_score', '') or '')
    cs_key = cs_match.group(0) if cs_match else 'N'

    ir_value = pw['IR'][loanAmountKey].get(ir_key, 0)
    rd_value = pw['RD'][loanAmountKey].get(rd_key, 0)
    w_type_coef = pw['w_type'][loanAmountKey].get(loan.get('nickname', ''), 1)
    dd_value = pw['DD'].get(dd_key, 0)
    cs_value = pw['CS'].get(cs_key, 0)

    # Global weights
    w = pw['w']
    coef = (
        ir_value * w['IR_score'] +
        rd_value * w['RD_score'] +
        dd_value * w['DD_score'] +
        cs_value * w['CS_score']
    )
    # Business weight
    w_business = pw['w_business'].get(loan.get('nickname', ''), 1)

    # Final score
    # loan['w_IR_score'] = w['IR_score']
    # loan['ir_value'] = ir_value
    # loan['w_RD_score'] = w['RD_score']
    # loan['rd_value'] = rd_value
    # loan['w_DD_score'] = w['DD_score']
    # loan['dd_value'] = dd_value
    # loan['w_CS_score'] = w['CS_score']
    # loan['cs_value'] = cs_value
    
    # loan['w_type_coef'] = w_type_coef
    # loan['coef'] = coef
    # loan['w_business'] = w_business
    loan['sortOrder'] = coef * w_type_coef * w_business


In [6]:

def update_with_la(records: List[Dict[str, Any]], la: float) -> List[Dict[str, Any]]:
    """
    Given desired loan amount 'la', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        # calculate monthly repayment
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den / 10  # change to Toman
        rec['deposit_amount'] = (la / rec.get('loan_coefficient', 1)) * 100
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec['deposit_amount'] > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec['loan_amount'] <= la_lim
    valid_records = [r for r in records if valid(r)]
    # return valid_records
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)

In [7]:
# df_all = pd.DataFrame(_records)
# df_all.head()

In [8]:
# df = pd.DataFrame(update_with_la(_records, la = 700_000_000))
# df.head(15)

In [9]:
# len(update_with_la(_records, la = 700_000_000))

In [10]:
def update_with_da(records: List[Dict[str, Any]], da: float) -> List[Dict[str, Any]]:
    """
    Given deposit amount 'da', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        coeff = rec.get('loan_coefficient', 0) / 100
        la = coeff * da
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den / 10
        rec['deposit_amount'] = da
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec['deposit_amount'] > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec['loan_amount'] <= la_lim

    valid_records = [r for r in records if valid(r)]
    # Sort descending by sortOrder
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)



In [11]:
# print(update_with_da(_records, da = 200_000_000))

In [12]:
# len(update_with_da(_records, da = 200_000_000))

In [13]:
def query_complex(
    scenarios: List[Dict[str, Any]],
    deposit_amount: Optional[float] = None,
    repayment_duration: Optional[int] = None,
    deposit_duration: Optional[int] = None,
    interest_rate: Optional[float] = None,
    credit_score: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Filter scenarios based on provided parameters.
    """
    matches = []
    for rec in scenarios:
        # deposit range: up to 1.6x
        cond_da = (deposit_amount is None or
                   rec.get('deposit_amount', 0) <= deposit_amount * 10_000_000 * 1.6)
        cond_rd = (repayment_duration is None or
                   rec.get('repayment_duration') == repayment_duration)
        cond_dep = (deposit_duration is None or
                    rec.get('deposit_duration') == deposit_duration)
        cond_ir = (interest_rate is None or
                   rec.get('interest_rate') == interest_rate)
        cs_field = rec.get('credit_score', '')
        cond_cs = (credit_score is None or
                   (credit_score in cs_field) or
                   (credit_score == 'N' and 'فاقد رتبه' in cs_field))
        if cond_da and cond_rd and cond_dep and cond_ir and cond_cs:
            matches.append(rec)
    return sorted(matches, key=lambda x: x.get('sortOrder', 0), reverse=True)




In [37]:

def get_query_params( _records, 
    deposit__amount: Optional[float] = None,
    repayment__duration: Optional[int] = None,
    deposit__duration: Optional[int] = None,
    interest__rate: Optional[float] = None,
    credit__score: Optional[str] = None,
    loan__amount: Optional[float] = None
) -> List[Dict[str, Any]]:

    if loan__amount:
        scenarios = update_with_la(_records, loan__amount)
    elif deposit__amount:
        scenarios = update_with_da(_records, deposit__amount)
    else:
        scenarios = _records

    report = query_complex(
        scenarios,
        deposit_amount=deposit__amount,
        repayment_duration=repayment__duration,
        deposit_duration=deposit__duration,
        interest_rate=interest__rate,
        credit_score=credit__score
    )

    return report

In [38]:
result = get_query_params( _records, 
    deposit__amount= 400_000_000,
    repayment__duration = None,
    deposit__duration = None,
    interest__rate = 23,
    credit__score = None,
    loan__amount = 700_000_000 )

In [39]:
len(result)

85

In [36]:
print(result)

[{'id': 2, 'nickname': 'بهان(بدون ضامن)', 'package_name': 'فرابانک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 1000000000, 'deposit_duration': 3, 'interest_rate': 23, 'repayment_duration': 60, 'loan_coefficient': 1000, 'credit_score': 'B', 'minimum_deposit_amount': 'nan', 'maximum_deposit_amount': 'nan', 'minimum_loan_amount': 100000000, 'guarantee': 'خیر', 'receiving_channel': 'غیر حضوری', 'loan_amount': 700000000, 'monthly_repayment': 1973332.976292618, 'deposit_amount': 70000000.0, 'sortOrder': 0.20334000000000002}, {'id': 1, 'nickname': 'بهان(بدون ضامن)', 'package_name': 'فرابانک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 1000000000, 'deposit_duration': 3, 'interest_rate': 23, 'repayment_duration': 60, 'loan_coefficient': 1000, 'credit_score': 'A', 'minimum_deposit_amount': 'nan', 'maximum_deposit_amount': 'nan', 'minimum_loan_amount': 100000000, 'guarantee': 'خیر', 'receiving_channel': 'غیر حضوری', 

In [32]:
pd.DataFrame(result).head(10)

,id,nickname,package_name,contract_type,granted_method,loan_amount_limit,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,minimum_deposit_amount,maximum_deposit_amount,minimum_loan_amount,guarantee,receiving_channel,loan_amount,monthly_repayment,deposit_amount,sortOrder
0,2,بهان(بدون ضامن),فرابانک,مرابحه,واریز به حساب,1000000000,3,23,60,1000,B,nan,nan,100000000,خیر,غیر حضوری,700000000,1.973333e+06,7.000000e+07,0.203340
1,1,بهان(بدون ضامن),فرابانک,مرابحه,واریز به حساب,1000000000,3,23,60,1000,A,nan,nan,100000000,خیر,غیر حضوری,700000000,1.973333e+06,7.000000e+07,0.194340
2,101,فرابانک,فرابانک,مرابحه,واریز به حساب,1500000000,3,23,36,1000,B,nan,nan,100000000,بله,غیر حضوری,700000000,2.709681e+06,7.000000e+07,0.122724
3,100,فرابانک,فرابانک,مرابحه,واریز به حساب,2000000000,3,23,36,1000,A,nan,nan,100000000,بله,غیر حضوری,700000000,2.709681e+06,7.000000e+07,0.117324
4,102,فرابانک,فرابانک,مرابحه,واریز به حساب,1000000000,3,23,36,1000,C و فاقد رتبه,nan,nan,100000000,بله,غیر حضوری,700000000,2.709681e+06,7.000000e+07,0.114624
5,35,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,1,23,12,50,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,700000000,6.585342e+06,1.400000e+09,0.061722
6,40,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,2,23,36,30,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,700000000,2.709681e+06,2.333333e+09,0.061362
7,41,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,2,23,36,35,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,700000000,2.709681e+06,2.000000e+09,0.061362
8,48,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,3,23,36,50,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,700000000,2.709681e+06,1.400000e+09,0.061362
9,49,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,3,23,36,60,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,700000000,2.709681e+06,1.166667e+09,0.061362
